# SPAR — 4 composites x 7 tiles, gross/net toggle, as-of 0Q

Fabric **Python** notebook (not PySpark). Pulls SPAR Engine results for four GIPS
composite ACCTs against each one's default benchmark, across seven saved SPAR components
("tiles"), on a gross / net / both basis, and lands them in `hbcm_datahub`.

```
grid = TILES x STRATEGIES x FEE_BASIS  ->  one SPAR calculation unit each
     = 7     x 4          x 2          =  56 units, submitted as 7 per-tile batches
```

## The tiles

| Tile | Window | Peer universe |
|---|---|---|
| `multi_horizon_returns` | inception → 0Q | — |
| `calendar_year_returns` | inception → 0Q | — |
| `cumulative_monthly` | inception → 0Q | — |
| `monthly_raw_returns` | inception → 0Q | — |
| `peer_multi_horizon` | inception → 0Q | **yes** |
| `peer_calendar_year` | inception → 0Q | **yes** |
| `risk_stats_3y` | −3Y → 0Q | — |
| `risk_stats_itd` *(planned)* | inception → 0Q | — |

Dates are **per tile**, not global — risk stats want a fixed 3-year window while the
cumulative and calendar-year tiles want inception-to-date. `startdate: "INCEPTION"`
resolves per-strategy from that ACCT's inception date, which is how the planned
since-inception risk variant reuses the same component id under a second key.

`universeid` is sent **only** for the two peer tiles. On the others it is omitted from
the payload entirely rather than sent as null, which would override the component's
saved universe with nothing.

## Fee basis

Each ACCT carries **both** a gross and a net return stream, so the toggle is
`SPARIdentifier.returntype` — one account id, two return types. Set
`SYMBOL_SUFFIX_MODE = True` for the alternative QR/`$$Performance.ofdb` layout where
gross and net are separate `_G` / `_N` symbols.

Return type values are often numeric codes, not the literal strings `"Gross"` / `"Net"`.
Cell 4a prints what the API reports per ACCT — take the values from there.

## What you must fill in before first run

All of it is workstation-sourced, all in Cell 3, and Cell 4 discovers every piece:

| Field | Discovery |
|---|---|
| `STRATEGIES[code]["acct"]` + `["prefix"]` | Cell 4a |
| `STRATEGIES[code]["returntype"]` (gross, net) | Cell 4a |
| `STRATEGIES[code]["benchmark"]` (id, prefix) | Cell 4e — **a wrong prefix returns a bare 400 with no message** |
| `STRATEGIES[code]["inception"]` | from the composite record |
| `STRATEGIES[code]["universe"]` | Cell 4c |
| `TILES[tile]["componentid"]` | Cell 4b |
| `TILES[tile]["frequency"]` | Cell 4d |

Cell 3 asserts that every peer tile has a universe and every `INCEPTION` tile has an
inception date, so bad config fails before an API call is spent.

## Versions — latest as of 2026-08-06

| Package | Version | Notes |
|---|---|---|
| `fds.sdk.SPAREngine` | **3.0.0** | released 2026-05-20 |
| `fds.sdk.utils` | 3.0.1 | OAuth helper |
| `fds.protobuf.stach.extensions` | 1.3.3 | STACH parsing |
| `deltalake` | 1.6.2 | OneLake write |

SPAREngine 3.0.0 is a **major bump for runtime hygiene, not an API redesign**. Per
FactSet's `BREAKING.md` (2026-05-20) every Python SDK was bumped together: Python
3.7/3.8/3.9 support dropped, and `urllib3` moved from `>=1.25.3,<2.1.0` to `>=2.7.0`.
Every model and method this notebook touches is unchanged between the 2.x docs and 3.0.0.

Minimum Python is now **3.10**. Fabric Python notebook kernels are 3.10 / 3.11 / 3.12
with **3.12 the default**, so any current kernel satisfies it — prefer 3.12, since 3.10
reaches end of support in October 2026.

> ⚠️ The SPAR SDK vendored under `code/python/SPAREngine/v3/` **in this repo is 2.0.3**
> (last synced 2025-07-21) and predates both the `universeid` field and
> `SPARPeerUniverseApi`. Verify any SPAR question against upstream `main`, not the local
> mirror. PA Engine is at **4.0.0** upstream and carries a second, separate breaking
> change (2026-07-21: required fields dropped from `PADateParameters`) that matters for
> the PA side of the pipeline but not for this notebook.

## Library setup — read this before scheduling

Do **not** rely on `%pip install` here. Per Microsoft, inline installs are *disabled by
default in pipeline runs* and *unsupported in reference runs*, and `%pip`-installed
libraries are not retained across runs. Attach the four packages above to a **Fabric
Environment** and bind this notebook to it.

For interactive first-run only:
```
%pip install fds.sdk.SPAREngine==3.0.0 fds.sdk.utils==3.0.1 \
             fds.protobuf.stach.extensions==1.3.3 deltalake==1.6.2
```

In [ ]:
# === Cell 1: credentials ===================================================
# HBCM_Config defines FACTSET_USER and FACTSET_APIKEY as plain strings.
# %run works in both interactive and pipeline mode; notebookutils.notebook.run() does not
# propagate variables, so keep this as %run.
%run HBCM_Config

In [ ]:
# === Cell 2: imports + API client ==========================================
import json, time, datetime as dt
import pandas as pd

import fds.sdk.SPAREngine
from fds.sdk.SPAREngine.api import (
    spar_calculations_api,
    spar_peer_universe_api,   # SDK >= 2.1; absent from the 2.0.3 copy vendored here
    accounts_api,
    components_api,
    benchmarks_api,
    frequencies_api,
)
from fds.sdk.SPAREngine.models import (
    SPARCalculationParametersRoot,
    SPARCalculationParameters,
    SPARIdentifier,
    SPARDateParameters,
    CalculationMeta,
)
from urllib3 import Retry   # SDK 3.0.0 requires urllib3 >= 2.7.0

# Fail loudly on a stale Environment rather than 400-ing later on universeid.
# Compare the major as an int — a string compare would rank "10.0.0" below "3.0.0".
SDK_VERSION = fds.sdk.SPAREngine.__version__
assert int(SDK_VERSION.split(".")[0]) >= 3, (
    f"fds.sdk.SPAREngine {SDK_VERSION} found; this notebook targets >=3.0.0. "
    "Check the bound Fabric Environment."
)
print("SPAREngine SDK", SDK_VERSION)

configuration = fds.sdk.SPAREngine.Configuration(
    username=FACTSET_USER,
    password=FACTSET_APIKEY,
)
# Preferred once an app-config.json exists in Key Vault:
#   from fds.sdk.utils.authentication import ConfidentialClient
#   configuration = fds.sdk.SPAREngine.Configuration(
#       fds_oauth_client=ConfidentialClient(str(config_path)))

# Retry server-side failures only. Never add 429 with a naive backoff — urllib3 already
# honours Retry-After for 429, and 4xx other than 429 will not fix themselves.
configuration.retries = Retry(
    total=3,
    status_forcelist=[500, 502, 503, 504],
    backoff_factor=2,
    allowed_methods=frozenset(["GET", "POST"]),
)

api_client = fds.sdk.SPAREngine.ApiClient(configuration)
calc_api = spar_calculations_api.SPARCalculationsApi(api_client)

In [ ]:
# === Cell 3: THE CONFIG BLOCK — the only cell you normally edit ============

CURRENCY = "USD"            # all GIPS series are USD-denominated
AS_OF = "0Q"                # most recent quarter end, FactSet relative-date grammar

def _prior_quarter_end(today=None):
    """Absolute mirror of AS_OF, used for labelling. Last completed calendar quarter."""
    d = today or dt.date.today()
    qe = dt.date(d.year, ((d.month - 1) // 3) * 3 + 1, 1) - dt.timedelta(days=1)
    return qe.strftime("%Y%m%d")

AS_OF_ABS = _prior_quarter_end()

# --- fee basis toggle ------------------------------------------------------
# Each ACCT carries BOTH a gross and a net return stream, so the toggle is
# SPARIdentifier.returntype — one account id, two return types. (Contrast the QR/OFDB
# layout where gross and net are separate GIPS_<STRAT>_G / _N symbols; that path is
# still available via SYMBOL_SUFFIX_MODE below.)
#
# returntype values go in STRATEGIES[...]["returntype"]. They are often numeric codes
# rather than the literal strings "Gross"/"Net" — Cell 4 prints what the API reports.
FEE_BASIS = "both"          # "gross" | "net" | "both"
SYMBOL_SUFFIX_MODE = False  # True => append _G/_N to the account id instead
SUFFIX = {"gross": "_G", "net": "_N"}

# --- the four composites ---------------------------------------------------
# TODO — one entry per ACCT. Every field here is workstation-sourced; a wrong `prefix`
# on either the account or the benchmark returns a bare 400 with no diagnostic.
STRATEGIES = {
    "<CODE1>": {
        "label":      "<Strategy name>",
        "acct":       "<ACCT identifier, no prefix>",
        "prefix":     "CLIENT:",
        "returntype": {"gross": "<gross returntype>", "net": "<net returntype>"},
        "benchmark":  {"id": "<TODO>", "prefix": "BENCH:", "returntype": None},
        "inception":  "<YYYYMMDD>",
        "universe":   "<peer universe id>",   # used only by the peer tiles
    },
    # "<CODE2>": {...}, "<CODE3>": {...}, "<CODE4>": {...}
}

# --- the tiles -------------------------------------------------------------
# A "tile" is a saved SPAR component. It fixes the statistic columns server-side; the
# POST body acts as a one-time override of the component's saved dates.
#
#   startdate       "INCEPTION" resolves per-strategy from STRATEGIES[...]["inception"],
#                   otherwise a relative token ("-3Y") or absolute YYYYMMDD.
#   needs_universe  True  -> universeid is sent, and the strategy must have one.
#                   False -> universeid omitted entirely (never sent as null).
#   frequency       Confirm against FrequenciesApi (Cell 4e). Calendar-year and
#                   multi-horizon tiles may want something other than Monthly.
#
# A component id may appear more than once under different variant keys — that is how
# the eventual "risk stats since inception" variant sits alongside the 3Y one.
TILES = {
    "multi_horizon_returns": {
        "componentid": "<TODO>", "startdate": "INCEPTION",
        "frequency": "Monthly", "needs_universe": False,
    },
    "calendar_year_returns": {
        "componentid": "<TODO>", "startdate": "INCEPTION",
        "frequency": "Monthly", "needs_universe": False,
    },
    "cumulative_monthly": {
        "componentid": "<TODO>", "startdate": "INCEPTION",
        "frequency": "Monthly", "needs_universe": False,
    },
    "monthly_raw_returns": {
        "componentid": "<TODO>", "startdate": "INCEPTION",
        "frequency": "Monthly", "needs_universe": False,
    },
    "peer_multi_horizon": {
        "componentid": "<TODO>", "startdate": "INCEPTION",
        "frequency": "Monthly", "needs_universe": True,
    },
    "peer_calendar_year": {
        "componentid": "<TODO>", "startdate": "INCEPTION",
        "frequency": "Monthly", "needs_universe": True,
    },
    "risk_stats_3y": {
        "componentid": "<TODO>", "startdate": "-3Y",
        "frequency": "Monthly", "needs_universe": False,
    },
    # Planned variant — same component, inception-to-date window:
    # "risk_stats_itd": {
    #     "componentid": "<same as risk_stats_3y>", "startdate": "INCEPTION",
    #     "frequency": "Monthly", "needs_universe": False,
    # },
}

SPAR_DOCUMENT = "Client:/SPAR/HBCM"    # for the component discovery cell only
PEER_UNIVERSE_CATEGORY = "Custom"      # for the peer universe discovery cell only

# --- execution -------------------------------------------------------------
# 7 tiles x 4 strategies x 2 bases = 56 units. Submitting all of them as one calculation
# is all-or-nothing and pushes the long-running window; batching per tile gives 8 units
# per calc, isolates failures to one tile, and stays clear of the ~5-10 concurrent-calc
# ceiling. Set False only for a small ad hoc run.
BATCH_PER_TILE = True

# --- OneLake target --------------------------------------------------------
WORKSPACE_ID = "1b9fac18-9d75-4437-ab6c-b6ba44ff46a8"   # HBCM - Production
LAKEHOUSE_ID = "7cdf13b1-4586-4a02-b8ff-72fcf6db1277"   # hbcm_datahub (schema-enabled)
ONELAKE = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"
TABLE_PATH = f"{ONELAKE}/Tables/factset/spar_composite_returns"
RAW_DIR = f"{ONELAKE}/Files/raw/spar"

BASES = ["gross", "net"] if FEE_BASIS == "both" else [FEE_BASIS]

# Fail fast on config that cannot possibly work, before burning an API call.
for _t, _cfg in TILES.items():
    if _cfg["needs_universe"]:
        _missing = [c for c, s in STRATEGIES.items() if not s.get("universe")]
        assert not _missing, f"tile {_t} needs a universe; missing for {_missing}"
    if _cfg["startdate"] == "INCEPTION":
        _missing = [c for c, s in STRATEGIES.items() if not s.get("inception")]
        assert not _missing, f"tile {_t} starts at INCEPTION; missing for {_missing}"

print(f"{len(TILES)} tiles x {len(STRATEGIES)} strategies x {len(BASES)} basis "
      f"= {len(TILES) * len(STRATEGIES) * len(BASES)} units"
      f"{f' in {len(TILES)} batches' if BATCH_PER_TILE else ' in 1 calculation'}")

In [ ]:
# === Cell 4: DISCOVERY — run interactively once, then leave it alone =======
# Resolves the TODOs in Cell 3. Not part of the scheduled path.

acct_api = accounts_api.AccountsApi(api_client)
comp_api = components_api.ComponentsApi(api_client)
bmk_api = benchmarks_api.BenchmarksApi(api_client)
peer_api = spar_peer_universe_api.SPARPeerUniverseApi(api_client)
freq_api = frequencies_api.FrequenciesApi(api_client)

# (a) Return types per ACCT -> STRATEGIES[...]["returntype"]
#     This is the authoritative answer on gross vs net. ReturnType carries (name, id);
#     the `id` is what goes in SPARIdentifier.returntype and is often a numeric code,
#     not the literal string "Net".
for code, s in STRATEGIES.items():
    path = s["acct"] if s["acct"].endswith(".ACCT") else f"{s['acct']}.ACCT"
    try:
        resp = acct_api.get_spar_returns_type(path)   # URL-encoded account path
        types = [(rt.get("name"), rt.get("id")) for rt in (resp.data.returns_type or [])]
        print(f"OK   {code:8s} {path:36s} -> {types}")
    except fds.sdk.SPAREngine.ApiException as e:
        print(f"FAIL {code:8s} {path:36s} -> {e.status} {e.body}")

# (b) Component ids -> TILES[...]["componentid"]
try:
    comps = comp_api.get_spar_components(document=SPAR_DOCUMENT)
    for cid, c in (comps.data or {}).items():
        print(f"component {cid}  {getattr(c, 'name', '')}  ({getattr(c, 'category', '')})")
except fds.sdk.SPAREngine.ApiException as e:
    print(f"components lookup failed: {e.status} {e.body}")

# (c) Peer universes -> STRATEGIES[...]["universe"] (needed by the two peer tiles).
#     `category` is required; `name` and `directory` are optional filters.
try:
    print(peer_api.get_list_of_peer_universe(PEER_UNIVERSE_CATEGORY))
except fds.sdk.SPAREngine.ApiException as e:
    print(f"peer universe lookup failed: {e.status} {e.body}")

# (d) Valid frequency ids -> TILES[...]["frequency"]. Calendar-year and multi-horizon
#     tiles may want something other than Monthly.
try:
    print(freq_api.get_spar_frequencies())
except fds.sdk.SPAREngine.ApiException as e:
    print(f"frequencies lookup failed: {e.status} {e.body}")

# (e) Confirm a benchmark id resolves and see its valid prefixes -> ["benchmark"]
#   print(bmk_api.get_spar_benchmark_by_id(id="<benchmark id>"))

In [ ]:
# === Cell 5: build the calculation units ==================================
# One unit per (tile, strategy, basis).
#
# Why not collapse the four composites into a single unit's `accounts` list? Because
# SPARCalculationParameters.benchmark is a SCALAR, and each composite has its own
# benchmark. Accounts can only share a unit if they share a benchmark. The flat shape
# also keeps one unit key == one result block, which is far easier to debug.

def account_identifier(code: str, basis: str) -> SPARIdentifier:
    s = STRATEGIES[code]
    if SYMBOL_SUFFIX_MODE:
        # Separate gross/net symbols (QR / $$Performance.ofdb layout).
        return SPARIdentifier(id=f"{s['acct']}{SUFFIX[basis]}", returntype=None,
                              prefix=s["prefix"])
    # One ACCT carrying both streams — pick the stream by return type.
    return SPARIdentifier(id=s["acct"], returntype=s["returntype"][basis],
                          prefix=s["prefix"])

def resolve_startdate(tile_cfg: dict, code: str) -> str:
    sd = tile_cfg["startdate"]
    return STRATEGIES[code]["inception"] if sd == "INCEPTION" else sd

def build_unit(tile_cfg: dict, code: str, basis: str) -> SPARCalculationParameters:
    s = STRATEGIES[code]
    bmk = s["benchmark"]
    kwargs = dict(
        componentid=tile_cfg["componentid"],
        accounts=[account_identifier(code, basis)],
        benchmark=SPARIdentifier(id=bmk["id"], returntype=bmk["returntype"],
                                 prefix=bmk["prefix"]),
        dates=SPARDateParameters(
            startdate=resolve_startdate(tile_cfg, code),
            enddate=AS_OF,
            frequency=tile_cfg["frequency"],
            # Explicit per-strategy inception dates are used instead of this flag, so
            # each unit's window is visible in the request rather than inferred.
            useeachportfolioinception=False,
        ),
        currencyisocode=CURRENCY,
    )
    # Send universeid only for the peer tiles. Passing None on a non-peer tile would
    # override the component's saved universe with nothing.
    if tile_cfg["needs_universe"]:
        kwargs["universeid"] = s["universe"]
    return SPARCalculationParameters(**kwargs)

def unit_key(tile_name: str, code: str, basis: str) -> str:
    return f"{tile_name}__{code}__{basis}"

UNIT_KEYS = {}          # unit_key -> (tile_name, strategy_code, basis)
batches = []            # list[(batch_label, SPARCalculationParametersRoot)]

def make_root(units: dict) -> SPARCalculationParametersRoot:
    return SPARCalculationParametersRoot(
        data=units,
        meta=CalculationMeta(
            contentorganization="SimplifiedRow",
            stach_content_organization="SimplifiedRow",
            contenttype="Json",
            format="JsonStach",
        ),
    )

all_units = {}
for tile_name, tile_cfg in TILES.items():
    tile_units = {}
    for code in STRATEGIES:
        for basis in BASES:
            key = unit_key(tile_name, code, basis)
            tile_units[key] = build_unit(tile_cfg, code, basis)
            UNIT_KEYS[key] = (tile_name, code, basis)
    all_units.update(tile_units)
    if BATCH_PER_TILE:
        batches.append((tile_name, make_root(tile_units)))

if not BATCH_PER_TILE:
    batches = [("all", make_root(all_units))]

print(f"{len(all_units)} units across {len(batches)} batch(es): "
      + ", ".join(f"{label}({len(r.data)})" for label, r in batches))

In [ ]:
# === Cell 6: submit + poll ================================================
# Multi-unit calculations ALWAYS return 202 regardless of the deadline header, so the
# polling loop is the normal path here, not the exception.

def run_spar(params_root, deadline=10, poll_interval=3, timeout=900):
    """Returns list[(unit_id, result_or_None, status)] for one calculation."""
    wrapper = calc_api.post_and_calculate(
        x_fact_set_api_long_running_deadline=deadline,
        spar_calculation_parameters_root=params_root,
    )
    code = wrapper.get_status_code()
    if code == 200:
        status_root = wrapper.get_response_200()
    elif code == 201:
        status_root = wrapper.get_response_201()
    elif code == 202:
        status_root = wrapper.get_response_202()
        calc_id = status_root.data.calculationid
        deadline_at = time.time() + timeout
        while True:
            if time.time() > deadline_at:
                calc_api.cancel_calculation_by_id(id=calc_id)
                raise TimeoutError(f"calc {calc_id} exceeded {timeout}s (cancelled)")
            poll = calc_api.get_calculation_status_by_id(id=calc_id)
            if poll.get_status_code() == 200:
                status_root = poll.get_response_200()
                break
            if poll.get_status_code() != 202:
                raise RuntimeError(f"unexpected poll status {poll.get_status_code()}")
            time.sleep(poll_interval)
    else:
        raise RuntimeError(f"unexpected submit status {code}")

    calc_id = status_root.data.calculationid
    out = []
    for unit_id, unit_status in (status_root.data.units or {}).items():
        st = getattr(unit_status, "status", None)
        if st != "Success":
            out.append((unit_id, None, st))
            continue
        res = calc_api.get_calculation_unit_result_by_id(id=calc_id, unit_id=unit_id)
        out.append((unit_id, res, st))
    return calc_id, out

results, calc_ids, batch_errors = [], {}, {}
for label, root in batches:
    try:
        cid, batch_results = run_spar(root)
        calc_ids[label] = cid
        results.extend(batch_results)
        n_bad = sum(1 for _, r, _ in batch_results if r is None)
        print(f"{label:24s} calc={cid} ok={len(batch_results) - n_bad} failed={n_bad}")
    except Exception as e:
        # One bad tile should not cost the other six. Record and continue.
        batch_errors[label] = repr(e)
        print(f"{label:24s} BATCH FAILED: {e!r}")

failed = [(u, s) for u, r, s in results if r is None]
for u, s in failed:
    print(f"  FAILED unit {u}: {s}")

# Log calc ids alongside X-DataDirect-Request-Key for any FactSet support ticket —
# use the *_with_http_info variants when you need the response headers.
print(f"\n{len(results) - len(failed)} usable units; "
      f"{len(failed)} failed units; {len(batch_errors)} failed batches")
assert results and not failed and not batch_errors, (
    "resolve failures before writing to the lakehouse — a partial snapshot is worse "
    "than none, because it looks complete downstream"
)

In [ ]:
# === Cell 7: raw landing (write-once audit copy) ==========================
# FactSet earns a raw layer: calls are slow and async, STACH reshaping is fiddly, and
# vendor restatements mean the same call replayed later does NOT return what it
# originally returned. That matters for GIPS and Marketing Rule substantiation.
# Files, not Delta — raw stays invisible to the SQL endpoint, which is correct.

asof_tag = AS_OF_ABS   # label raw by resolved quarter end, never by a relative token
for unit_id, res, _ in results:
    notebookutils.fs.put(
        f"{RAW_DIR}/asof={asof_tag}/{unit_id}.json",
        json.dumps(res.to_dict(), default=str),
        True,
    )
print(f"landed {len(results)} raw payloads under {RAW_DIR}/asof={asof_tag}/")

In [ ]:
# === Cell 8: STACH -> tidy DataFrame ======================================
from fds.protobuf.stach.extensions.StachExtensionFactory import StachExtensionFactory
from fds.protobuf.stach.extensions.StachVersion import StachVersion

def stach_to_dataframes(api_response):
    """STACH v2 payload -> list[DataFrame], one per table."""
    ext = StachExtensionFactory.get_stach_extension(StachVersion.V2)
    tables = ext.convert(json.dumps(api_response.to_dict(), default=str))
    return [pd.DataFrame(t.data, columns=t.columns) for t in tables]

frames = []
for unit_id, res, _ in results:
    tile_name, code, basis = UNIT_KEYS[unit_id]
    s, tile_cfg = STRATEGIES[code], TILES[tile_name]
    for i, df in enumerate(stach_to_dataframes(res)):
        df = df.copy()
        # Provenance first, so the grain is legible without joining anything.
        for pos, (col, val) in enumerate([
            ("asof_date",     asof_tag),
            ("tile",          tile_name),
            ("strategy_code", code),
            ("strategy",      s["label"]),
            ("fee_basis",     basis),
            ("account_id",    s["acct"]),
            ("benchmark_id",  s["benchmark"]["id"]),
            ("universe_id",   s["universe"] if tile_cfg["needs_universe"] else None),
            ("start_date",    resolve_startdate(tile_cfg, code)),
            ("frequency",     tile_cfg["frequency"]),
            ("table_ix",      i),
        ]):
            df.insert(pos, col, val)
        frames.append(df)

tidy = pd.concat(frames, ignore_index=True)
tidy.columns = [str(c).strip().replace(" ", "_").lower() for c in tidy.columns]
tidy = tidy.astype({c: "string" for c in tidy.select_dtypes("object").columns})

print(tidy.shape)
print(tidy.groupby(["tile", "fee_basis"], dropna=False).size())
display(tidy.head(20))

In [ ]:
# === Cell 9: idempotent write to hbcm_datahub =============================
# delete-then-append on asof_date, so a pipeline retry or manual re-run replaces the
# snapshot instead of doubling it. mode="append" alone would silently duplicate.
from deltalake import DeltaTable, write_deltalake

try:
    DeltaTable(TABLE_PATH).delete(f"asof_date = '{asof_tag}'")
    mode = "append"
except Exception:
    mode = "overwrite"          # table does not exist yet

write_deltalake(TABLE_PATH, tidy, mode=mode, schema_mode="merge")
print(f"wrote {len(tidy)} rows to factset.spar_composite_returns (mode={mode})")

# No partitioning: this table is kilobytes. Partitioning a small table costs more in
# metadata than it saves in pruning.
# Re-frame any Direct Lake semantic model BEFORE running VACUUM — vacuuming files a
# framed model still points at gives users query errors on missing files.
# Order is always: write -> frame -> vacuum.

## Validation checklist

- [ ] Cell 2's version assert passed, so the Environment really is on SDK >= 3.0.0.
- [ ] 56 units expected (7 x 4 x 2); all 7 batches succeeded and no unit failed.
- [ ] Gross exceeds net for every strategy and period. If gross == net, the two
      `returntype` values in Cell 3 aren't resolving to distinct streams — re-read Cell 4a.
- [ ] The two peer tiles carry a non-null `universe_id`; the other five carry null.
- [ ] `start_date` on inception tiles matches each composite's actual inception, and
      `-3Y` only on the risk tile.
- [ ] `asof_date` matches the quarter end you expect. `AS_OF = "0Q"` resolves
      server-side; `AS_OF_ABS` is what gets written. If those disagree, the labels lie.
- [ ] Returns tie to the composite performance report — that report, not SPAR, is the
      GIPS authority.

## Known sharp edges

| Symptom | Cause |
|---|---|
| 400, no detail | Wrong `prefix` on account or benchmark. The single most common failure. |
| 400 on dates | `"0Q"` or the tile's frequency unsupported for that component — check Cell 4d. |
| 400 mentioning `universeid` | Environment is on an SDK older than 2.1 — the field didn't exist. Cell 2's assert should catch this first. |
| `ImportError` on `spar_peer_universe_api` | Same cause: stale SDK. |
| Gross and net identical | Both `returntype` values resolving to the same stream. |
| Peer tile returns no percentiles | `universe` id wrong, or the component's saved universe conflicts with the override. |
| 404 fetching a result | Calculation id expired (TTL is hours). Resubmit that batch. |
| 429 | Concurrency ceiling, typically 5–10 concurrent calcs per user. Batching per tile is already the mitigation; don't set `BATCH_PER_TILE = False` on the full grid. |
| One tile empty, others fine | That component's saved date range or grouping conflicts with the override. Per-tile batching means it can't take the rest down. |
| Works interactively, fails in pipeline | `%pip` install — bind a Fabric Environment instead. |

## Sources

Verified against **upstream `FactSet/enterprise-sdk` `main`** (SPAREngine v3, SDK 3.0.0)
— `SPARCalculationsApi.md`, `SPARCalculationParameters.md`, `SPARIdentifier.md`,
`SPARDateParameters.md`, `CalculationMeta.md`, `AccountsApi.md`
(`get_spar_returns_type`), `SPARPeerUniverseApi.md`, `FrequenciesApi.md`,
`ReturnType.md`, plus `BREAKING.md` for the 2026-05-20 Python-SDK bump.

Not against `code/python/SPAREngine/v3/` in this repo, which is pinned at 2.0.3
(2025-07-21) and lacks `universeid` and `SPARPeerUniverseApi`.

Microsoft Learn — [notebook limitations](https://learn.microsoft.com/fabric/data-engineering/notebook-limitation),
[%run](https://learn.microsoft.com/fabric/data-engineering/author-execute-notebook#run-notebooks),
[Python kernel lifecycle](https://learn.microsoft.com/fabric/data-engineering/python-notebook-runtime-lifecycle),
[pandas to lakehouse](https://learn.microsoft.com/fabric/data-engineering/lakehouse-notebook-load-data#load-data-with-pandas-api).